# Wind Power — Feature Engineering

Goal: prepare the wind SCADA data for forecasting turbine power **1 hour ahead**.

Every feature uses only information available at prediction time (the current reading and past readings), so no future data leaks into the model.

Steps:
1. Load and rename columns
2. Check for missing timestamps and fill short gaps
3. Handle negative power readings
4. Create the target (power 1 hour ahead)
5. Create lag, rolling, and cyclical time features
6. Drop incomplete rows and save the processed dataset

In [1]:
import pandas as pd
import numpy as np

# Load raw wind data and parse the timestamp (format confirmed during EDA: DD MM YYYY HH:MM)
wind_df = pd.read_csv('../data/raw/wind/T1.csv')
wind_df['Date/Time'] = pd.to_datetime(wind_df['Date/Time'], format='%d %m %Y %H:%M')

# Rename columns to short, code-friendly names
wind_df = wind_df.rename(columns={
    'Date/Time': 'timestamp',
    'LV ActivePower (kW)': 'power',
    'Wind Speed (m/s)': 'wind_speed',
    'Theoretical_Power_Curve (KWh)': 'theoretical_power',
    'Wind Direction (°)': 'wind_direction'
})

wind_df.head()

,timestamp,power,wind_speed,theoretical_power,wind_direction
0,2018-01-01 00:00:00,380.047791,5.311336,416.328908,259.994904
1,2018-01-01 00:10:00,453.769196,5.672167,519.917511,268.641113
2,2018-01-01 00:20:00,306.376587,5.216037,390.900016,272.564789
3,2018-01-01 00:30:00,419.645905,5.659674,516.127569,271.258087
4,2018-01-01 00:40:00,380.650696,5.577941,491.702972,265.674286


## Check for Missing Timestamps

The turbine records every 10 minutes, so a full year should have 52,560 rows. Lag features assume evenly spaced rows: if some timestamps are missing, "1 hour ago" would silently point to the wrong row. So we first check how many are missing.

In [2]:
# Remove any duplicate timestamps, sort chronologically, and use the timestamp as the index
wind_df = wind_df.drop_duplicates(subset='timestamp').sort_values('timestamp').set_index('timestamp')

# Build the complete 10-minute timeline from the first to the last reading
full_range = pd.date_range(wind_df.index.min(), wind_df.index.max(), freq='10min')

print('Expected rows:     ', len(full_range))
print('Actual rows:       ', len(wind_df))
print('Missing timestamps:', len(full_range) - len(wind_df))

Expected rows:      52560
Actual rows:        50530
Missing timestamps: 2030


## Fill Short Gaps

We insert the missing timestamps so every row is exactly 10 minutes apart. Gaps of up to 1 hour are filled by interpolating between neighbouring readings. Longer gaps are left empty, because guessing hours of turbine behaviour would create fake data. Rows affected by long gaps are dropped at the end.

In [3]:
# Insert missing timestamps (their values start as NaN)
wind_df = wind_df.reindex(full_range)
wind_df.index.name = 'timestamp'

# Fill gaps of up to 6 steps (1 hour) using time-based interpolation; longer gaps stay NaN
# (Wind direction wraps around at 360°, so interpolation can be slightly off across that
#  boundary — acceptable for gaps of an hour or less)
wind_df = wind_df.interpolate(method='time', limit=6)

print('Rows still missing after filling short gaps:', wind_df['power'].isna().sum())

Rows still missing after filling short gaps: 1906


## Handle Negative Power

EDA showed a few small negative power values. These happen when the turbine draws a little power from the grid while idling. For forecasting *generation*, we clip them to 0.

In [4]:
print('Negative power readings:', (wind_df['power'] < 0).sum())

# Clip negative values to 0 — the turbine is not generating in those moments
wind_df['power'] = wind_df['power'].clip(lower=0)

Negative power readings: 57


## Create the Target: Power 1 Hour Ahead

We want to predict power 1 hour into the future. With 10-minute data, 1 hour = 6 steps, so the target for each row is the power value 6 rows later.

In [5]:
HORIZON = 6  # 6 steps x 10 min = 1 hour ahead

# shift(-6) moves future values up, so each row holds the power 1 hour after its timestamp
wind_df['target_power_1h'] = wind_df['power'].shift(-HORIZON)

## Lag Features

Lag features give the model recent history: what the power and wind speed were 10 minutes ago, 1 hour ago, 24 hours ago, and so on. The current readings (`power`, `wind_speed`) are also features, because they're known at prediction time.

In [6]:
# Power lags: 10 min, 20 min, 30 min, 1 h, 2 h, and 24 h ago
for lag in [1, 2, 3, 6, 12, 144]:
    wind_df[f'power_lag_{lag}'] = wind_df['power'].shift(lag)

# Wind speed lags: 10 min, 30 min, and 1 h ago
for lag in [1, 3, 6]:
    wind_df[f'wind_speed_lag_{lag}'] = wind_df['wind_speed'].shift(lag)

## Rolling Features

Rolling features summarise recent behaviour over a window: the average level (mean) and how unstable it has been (standard deviation) over the past 1 and 3 hours. Wind is noisy, so these help the model separate a steady trend from a brief gust.

In [7]:
# Windows: 6 steps = past 1 hour, 18 steps = past 3 hours (each includes the current reading)
for window in [6, 18]:
    wind_df[f'power_roll_mean_{window}'] = wind_df['power'].rolling(window).mean()
    wind_df[f'power_roll_std_{window}'] = wind_df['power'].rolling(window).std()
    wind_df[f'wind_speed_roll_mean_{window}'] = wind_df['wind_speed'].rolling(window).mean()

## Cyclical Encoding (Wind Direction and Time)

Wind direction, hour of day, and month are circular: 359° is next to 1°, and 23:00 is next to 00:00. Using raw numbers would make them look far apart. Converting each into a sine and cosine pair places them on a circle, so neighbouring values stay close together.

This also helps the time-based split: December's month encoding is close to January's, which the model has seen during training.

In [8]:
# Wind direction (degrees) -> sine/cosine
rad = np.deg2rad(wind_df['wind_direction'])
wind_df['wind_dir_sin'] = np.sin(rad)
wind_df['wind_dir_cos'] = np.cos(rad)

# Hour of day (including minutes, e.g. 14:30 -> 14.5) -> sine/cosine over a 24-hour cycle
hour = wind_df.index.hour + wind_df.index.minute / 60
wind_df['hour_sin'] = np.sin(2 * np.pi * hour / 24)
wind_df['hour_cos'] = np.cos(2 * np.pi * hour / 24)

# Month -> sine/cosine over a 12-month cycle
month = wind_df.index.month
wind_df['month_sin'] = np.sin(2 * np.pi * month / 12)
wind_df['month_cos'] = np.cos(2 * np.pi * month / 12)

## Drop Unused Columns and Incomplete Rows

- `wind_direction` is replaced by its sine/cosine versions.
- `theoretical_power` is dropped from the forecasting features. It's a fixed function of wind speed, so it adds no new information (it may be used later in a stretch goal).
- Rows with missing values are dropped: the first 24 hours (no 24-hour lag yet), the last hour (no target yet), and rows around long data gaps.

In [9]:
wind_df = wind_df.drop(columns=['wind_direction', 'theoretical_power'])

rows_before = len(wind_df)
wind_df = wind_df.dropna()
print(f'Dropped {rows_before - len(wind_df)} incomplete rows; {len(wind_df)} rows remain')

wind_df.head()

Dropped 2854 incomplete rows; 49706 rows remain


,power,wind_speed,target_power_1h,power_lag_1,power_lag_2,power_lag_3,power_lag_6,power_lag_12,power_lag_144,wind_speed_lag_1,...,wind_speed_roll_mean_6,power_roll_mean_18,power_roll_std_18,wind_speed_roll_mean_18,wind_dir_sin,wind_dir_cos,hour_sin,hour_cos,month_sin,month_cos
timestamp,,,,,,,,,,,,,,,,,,,,,
2018-01-02 00:00:00,3377.649902,12.70539,3467.808105,3375.903076,3369.812988,3373.969971,3570.079102,3479.407959,380.047791,12.84406,...,12.844860,3509.203098,89.324099,12.890098,-0.276345,-0.961058,0.000000,1.000000,0.5,0.866025
2018-01-02 00:10:00,3374.423096,12.18336,3460.677002,3377.649902,3375.903076,3369.812988,3597.604004,3472.333008,453.769196,12.70539,...,12.784922,3496.597276,91.541653,12.826984,-0.304763,-0.952428,0.043619,0.999048,0.5,0.866025
2018-01-02 00:20:00,3401.081055,12.20514,3589.443115,3374.423096,3377.649902,3375.903076,3441.025879,3491.510986,306.376587,12.18336,...,12.706445,3485.300334,89.985945,12.776074,-0.289179,-0.957275,0.087156,0.996195,0.5,0.866025
2018-01-02 00:30:00,3469.412109,12.60188,3527.718994,3401.081055,3374.423096,3377.649902,3373.969971,3536.989014,419.645905,12.20514,...,12.639583,3477.801500,84.961802,12.740876,-0.238800,-0.971069,0.130526,0.991445,0.5,0.866025
2018-01-02 00:40:00,3511.809082,12.84455,3561.267090,3469.412109,3401.081055,3374.423096,3369.812988,3528.658936,380.650696,12.60188,...,12.564063,3472.661011,79.477724,12.719134,-0.247787,-0.968815,0.173648,0.984808,0.5,0.866025


## Save Processed Dataset

The feature table is saved to `data/processed/` for the modeling notebooks. The time-based train/test split happens in the modeling notebook, not here.

In [10]:
wind_df.to_csv('../data/processed/wind_features.csv')

print('Shape:', wind_df.shape)
print('Columns:', wind_df.columns.tolist())

Shape: (49706, 24)
Columns: ['power', 'wind_speed', 'target_power_1h', 'power_lag_1', 'power_lag_2', 'power_lag_3', 'power_lag_6', 'power_lag_12', 'power_lag_144', 'wind_speed_lag_1', 'wind_speed_lag_3', 'wind_speed_lag_6', 'power_roll_mean_6', 'power_roll_std_6', 'wind_speed_roll_mean_6', 'power_roll_mean_18', 'power_roll_std_18', 'wind_speed_roll_mean_18', 'wind_dir_sin', 'wind_dir_cos', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos']
